In [44]:
import os
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.utils import shuffle
from scipy.signal import resample
import matplotlib.pyplot as plt
import json
import warnings
import mne
from mne.preprocessing import ICA
warnings.filterwarnings("ignore")
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [45]:
SAMPLE_RATE = 200  # fs
# baseline correction according to paper, tmin and tmax was observed from the data
TASK_ID_1 = 0  # auditory oddball
TMIN_1, TMAX_1 = -0.2, 0.8
BASELINE_1 = (-0.2, 0)   

TASK_ID_2 = 1   # Visual Oddball
TMIN_2, TMAX_2 = -0.2, 0.8
BASELINE_2 = (-0.2, 0)

TASK_ID_3 = 10   # Visual Search
TMIN_3, TMAX_3 = -0.2, 0.8
BASELINE_3 = (-0.2, 0)

TASK_ID_4 = 5   # Flanker Task
TMIN_4, TMAX_4 = -0.2, 0.8
BASELINE_4 = (-0.2, 0)

sub_folder_path = str(SAMPLE_RATE) + 'Hz'
sub_folder_path

'200Hz'

## Load participants.tsv file

In [46]:
# root dir
root = "CESCA/"
# root = "Simon-conflict Task/ds004580-download/"
participants_path = os.path.join(root, 'participants.tsv')
participants = pd.read_csv(participants_path, sep='\t')
participants

,participant_id,Age_Grouped,Gender,Handedness,Highest_Edu_Grouped,Highest_Adult_Edu_Grouped,PassiveAuditoryOddballEEG,PassiveAuditoryOddballEEG_missingness_reason,VisualOddballEEG,VisualOddballEEG_missingness_reason,...,asrs16,asrs17,asrs18,asrs1_binary,asrs2_binary,asrs3_binary,asrs4_binary,asrs5_binary,asrs6_binary,ADHD_binary
0,sub-001,18 to 22,female,right,High School,Some College or Associate Degree,1,NaN,1,NaN,...,sometimes,rarely,rarely,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,sub-002,23 to 26,male,right,Bachelor's Degree or Higher,Some College or Associate Degree,0,Task not administered,1,NaN,...,never,never,never,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,sub-003,18 to 22,female,right,High School,Bachelor's Degree or Higher,0,Task not administered,1,NaN,...,rarely,rarely,sometimes,1.0,0.0,1.0,1.0,0.0,1.0,1.0
3,sub-004,18 to 22,female,right,High School,Bachelor's Degree or Higher,0,Task not administered,1,NaN,...,sometimes,rarely,never,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,sub-005,18 to 22,male,right,Some College or Associate Degree,High School or Less,0,Task not administered,1,NaN,...,rarely,never,never,0.0,0.0,1.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,sub-123,18 to 22,male,right,Some College or Associate Degree,Bachelor's Degree or Higher,1,NaN,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
123,sub-124,18 to 22,male,right,Some College or Associate Degree,Bachelor's Degree or Higher,1,NaN,1,NaN,...,rarely,never,rarely,1.0,0.0,0.0,1.0,0.0,1.0,0.0
124,sub-125,18 to 22,male,"ambidextrous (i.e., both right and left)",Some College or Associate Degree,High School or Less,1,NaN,1,NaN,...,sometimes,sometimes,sometimes,1.0,1.0,1.0,0.0,0.0,0.0,0.0
125,sub-126,18 to 22,male,right,Some College or Associate Degree,Bachelor's Degree or Higher,1,NaN,0,Participant-related issue,...,sometimes,rarely,rarely,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [47]:
# get Highest_Adult_Edu_Grouped info
unique_parent_degree = list(participants['Highest_Adult_Edu_Grouped'].unique())
unique_parent_degree

['Some College or Associate Degree',
 "Bachelor's Degree or Higher",
 'High School or Less']

In [48]:
degree_map = {'High School or Less':0, 'Some College or Associate Degree':1, "Bachelor's Degree or Higher":2}
degree_map

{'High School or Less': 0,
 'Some College or Associate Degree': 1,
 "Bachelor's Degree or Higher": 2}

In [49]:
participants[participants['participant_id']=='sub-001']['Highest_Adult_Edu_Grouped']

0    Some College or Associate Degree
Name: Highest_Adult_Edu_Grouped, dtype: object

## Find if there are bad channels, check sampling frequency and data shape to avoid inconsistency

In [50]:
# Test for bad channels, sampling freq and shape
"""bad_channel_list, sampling_freq_list, data_shape_list = [], [], []
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        for file in os.listdir(sub_path):
            if 'auditoryoddball_eeg.vhdr' in file:
            # if 'visualoddball_eeg.vhdr' in file:
                file_path = os.path.join(sub_path, file)
                print(file_path)
                raw = mne.io.read_raw_brainvision(file_path, preload=True)                
                bad_channel = raw.info['bads']
                bad_channel_list.append(bad_channel)
                # get sampling frequency
                sampling_freq = raw.info['sfreq']
                sampling_freq_list.append(sampling_freq)
                # get eeg data
                data = raw.get_data()
                data_shape = data.shape
                data_shape_list.append(data_shape)"""

"bad_channel_list, sampling_freq_list, data_shape_list = [], [], []\nfor sub in os.listdir(root):\n    if 'sub-' in sub:\n        sub_path = os.path.join(root, sub, 'eeg/')\n        for file in os.listdir(sub_path):\n            if 'auditoryoddball_eeg.vhdr' in file:\n            # if 'visualoddball_eeg.vhdr' in file:\n                file_path = os.path.join(sub_path, file)\n                print(file_path)\n                raw = mne.io.read_raw_brainvision(file_path, preload=True)                \n                bad_channel = raw.info['bads']\n                bad_channel_list.append(bad_channel)\n                # get sampling frequency\n                sampling_freq = raw.info['sfreq']\n                sampling_freq_list.append(sampling_freq)\n                # get eeg data\n                data = raw.get_data()\n                data_shape = data.shape\n                data_shape_list.append(data_shape)"

In [51]:
"""from collections import Counter

print(bad_channel_list)
print(data_shape_list[0])
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))"""

'from collections import Counter\n\nprint(bad_channel_list)\nprint(data_shape_list[0])\nprint("Channel number counter:", Counter(i[0] for i in data_shape_list))\nprint("Sampling rate counter:", Counter(sampling_freq_list))'

## Pick common channels

In [52]:
# channel number not consistent, take the common channels

"""common_channels = []
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        # print(sub_path)
        for file in os.listdir(sub_path):
            if 'auditoryoddball_eeg.vhdr' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_brainvision(file_path, preload=True)
                
                current_channels = set(raw.info['ch_names'])
                if not common_channels:
                    common_channels = current_channels
                else:
                    common_channels &= current_channels
common_channels = list(common_channels)
print(common_channels)
print("Common channels number: ", len(common_channels))"""

'common_channels = []\nfor sub in os.listdir(root):\n    if \'sub-\' in sub:\n        sub_path = os.path.join(root, sub, \'eeg/\')\n        # print(sub_path)\n        for file in os.listdir(sub_path):\n            if \'auditoryoddball_eeg.vhdr\' in file:\n                file_path = os.path.join(sub_path, file)\n                raw = mne.io.read_raw_brainvision(file_path, preload=True)\n                \n                current_channels = set(raw.info[\'ch_names\'])\n                if not common_channels:\n                    common_channels = current_channels\n                else:\n                    common_channels &= current_channels\ncommon_channels = list(common_channels)\nprint(common_channels)\nprint("Common channels number: ", len(common_channels))'

In [53]:
common_channels = ['F7', 'FC1', 'P8', 'FC5', 'P4', 'CP1', 'CP6', 'C4', 'LM', 'P7', 'Oz', 'F3', 'FC2', 'CP5', 'Fp2', 'HER', 'HEL', 'O2', 'Pz', 'T7', 'RM', 'P3', 'FC6', 'F4', 'Fp1', 'O1', 'C3', 'VER', 'CP2', 'F8', 'Fz']

In [54]:
# remove unreliable electrodes 
common_channels = [ch for ch in common_channels if ch not in ['LM', 'RM', 'HER', 'HEL', 'VER']]
print(common_channels)
print("Common channels number: ", len(common_channels))

['F7', 'FC1', 'P8', 'FC5', 'P4', 'CP1', 'CP6', 'C4', 'P7', 'Oz', 'F3', 'FC2', 'CP5', 'Fp2', 'O2', 'Pz', 'T7', 'P3', 'FC6', 'F4', 'Fp1', 'O1', 'C3', 'CP2', 'F8', 'Fz']
Common channels number:  26


## Data preprocessing and segmentation

In [55]:
def data_preprocessing(
    raw: mne.io.Raw,
    common_channels: list,
    sample_rate: int = 250,
    notch_freq: float = 60.0,
    l_freq: float = 0.5,
    h_freq: float = 40.0,
    do_bad_interp: bool = True,
    verbose: bool = True,
):
    """
    Preprocessing steps ：
      1) choose common channels and reorder
      2) Set Montage 
      3) 60 Hz Notch（before band pass）
      4) bandpass filter（default 0.5–40 Hz）
      5) interpolate bad channels（if do_bad_interp is True）
      6) re-reference to average
      7) ICA（在 1 Hz 高通的副本上拟合，自动剔除眼动/肌电等分量，需 mne-icalabel）
      8) downsample to 250 Hz
    """
    
    # 1. select common channels and reorder to given order
    keep = [ch for ch in common_channels if ch in raw.ch_names]
    raw.pick_channels(keep)
    raw.reorder_channels(keep)
    if verbose:
        print(f"✔ Step 1: Picked common channels ({len(keep)}): {keep}")
        
    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1020'))
    if verbose:
        print("✔ Step 2, Montage set: 'standard_1020'.")
        
    # 3. Notch（工频）
    if notch_freq is not None:
        raw.notch_filter(freqs=[notch_freq], picks="eeg", verbose=False)
        if verbose:
            print(f"✔ Step 3: Notch @ {notch_freq} Hz")
        
    # 4. Bandpass Filter (0.5–40 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass {l_freq}–{h_freq} Hz")
        
    # 5. Interpolate bad channels
    if do_bad_interp and raw.info.get("bads"):
        raw.interpolate_bads(reset_bads=True, verbose=False)
        if verbose:
            print(f"✔ Step 5: Interpolated bads: {raw.info.get('bads', [])}")
    else:
        if verbose:
            print("ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)")
            
    # 6) Re-reference to average
    raw.set_eeg_reference("average", verbose=False)
    if verbose:
        print("✔ Step 6: Average reference")
    
    print()
    # 7) ICA (fit ICLabel on 1 Hz high-pass filtered copy, then apply to original)
    raw_for_ica = raw.copy().filter(l_freq=1.0, h_freq=None, picks="eeg", verbose=False)
    ica = ICA(n_components=None, method="fastica", random_state=97, max_iter="auto")
    ica.fit(raw_for_ica)

    excluded = []
    if label_components is not None:
        try:
            ic_labels = label_components(raw_for_ica, ica, method="iclabel")
            labels = ic_labels["labels"]
            probs = ic_labels["y_pred_proba"]  # (n_comp, n_classes)
            thresholds = {
                "eye blink": 0.7,
                "muscle artifact": 0.6,
                "heart beat": 0.5,
                "line noise": 0.8,
                "channel noise": 0.9,
            }
            for i, lab in enumerate(labels):
                if lab in thresholds:
                    if probs is not None:
                        p = probs[i].max()
                    else:
                        p = 1.0
                    if p >= thresholds[lab]:
                        excluded.append(i)
        except Exception as e:
            if verbose:
                print(f"⚠ ICLabel failed ({e}). Skipping auto exclusion.")
    else:
        if verbose:
            print("ℹ ICLabel not available; fitted ICA but no auto component exclusion.")

    if excluded:
        ica.exclude = sorted(set(excluded))
        raw = ica.apply(raw.copy())
        if verbose:
            print(f"✔ Step 7: ICA applied. Excluded comps: {ica.exclude}")
    else:
        if verbose:
            print("ℹ Step 7: No ICA components excluded.")

    # 8) downsample to 250 Hz
    if raw.info["sfreq"] != sample_rate:
        raw.resample(sample_rate, npad="auto", verbose=False)
    if verbose:
        print(f"✔ Step 8: Resampled to {sample_rate} Hz")
        
    return raw

In [56]:
Auditory_oddball_mapping_dict = { # According to the dataset DOI, S180 represents Pre-standard stimulus, S 80 represents Standard stimulus, S 70 represents Deviant stimulus
    "S 80" : 0,  # Standard stimulus
    "S 70" : 1   # Deviant stimulus
}

Visual_oddball_mapping_dict = {
    # Standard stimulation (non-target)
    "S 12": 0, "S 13": 0, "S 14": 0, "S 15": 0,
    "S 21": 0, "S 23": 0, "S 24": 0, "S 25": 0,
    "S 31": 0, "S 32": 0, "S 34": 0, "S 35": 0,
    "S 41": 0, "S 42": 0, "S 43": 0, "S 45": 0,
    "S 51": 0, "S 52": 0, "S 53": 0, "S 54": 0,
    
    # Oddball stimulation (target)
    "S 11": 1, "S 22": 1, "S 33": 1, "S 44": 1, "S 55": 1,   # according to GPT explanation
    
    # Response events
    "S201": 2,  # Correct response
    "S202": 3   # Incorrect response
}

Visual_search_mapping_dict = {
    "S111": 0, "S112": 0, "S211": 0, "S212": 0,  # target location left
    "S121": 1, "S122": 1, "S221": 1, "S222": 1,  # target location right
    
    "S201": 2,   # Correct response
    "S202": 3    # Incorrect response
}

Flanker_mapping_dict = {
    "S 11": 0, "S 12": 0,  # compatible
    "S 21": 1, "S 22": 1,  # incompatible
    
    "S111": 2, "S121": 2, "S212": 2, "S222": 2,  # Correct response
    "S112": 3, "S122": 3, "S211": 3,  "S221": 3,  # Incorrect response
}


def epoch_and_make_xy(
    raw: mne.io.Raw,
    events_tsv_path: str,
    tmin: float = -0.2,
    tmax: float = 0.8,
    baseline=(-0.2, 0),               # e.g, (-0.5, 0)   baseline correction from -0.5 to 0 s
    task_id: int = 0,            # assigned task ID, 0 for auditory oddball tasks, 1 for visual oddball task
    subject_id: int = 1,     
    parent_degree: int = 0,      # ['High School or Less', 'Some College or Associate Degree', 'Bachelor's Degree or Higher'] -> (0, 1, 2)
):
    
    ev = pd.read_csv(events_tsv_path, sep="\t")
    
    mapping_dict ={}
    # Remove useless data at the beginning
    if task_id == 0:
        mapping_dict = Auditory_oddball_mapping_dict
        ev = ev[~ev['value'].isin(['S  1', 'S  2', 'S180', 'boundary', 'Buffer Overflow'])]  # remove irrelevant event 
    if task_id == 1:
        mapping_dict = Visual_oddball_mapping_dict
        ev = ev[~ev['value'].isin(['boundary', 'Buffer Overflow'])]
        ev["code"] = ev['value'].map(mapping_dict).astype(int)
        ev = ev[ev["code"].isin([0, 1])].reset_index(drop=True)  # using visual stimulus (both target and non-target) as time-lock
    if task_id == 10:
        mapping_dict = Visual_search_mapping_dict
        ev = ev[~ev['value'].isin(['boundary', 'Buffer Overflow'])]
        ev["code"] = ev['value'].map(mapping_dict).astype(int)
        ev = ev[ev["code"].isin([0, 1])].reset_index(drop=True)  # using visual stimulus (both left and right) as time-lock
    if task_id == 5:
        mapping_dict = Flanker_mapping_dict 
        ev = ev[~ev['value'].isin(['S  1', 'S  2', 'boundary', 'Buffer Overflow'])]  # remove irrelevant event 
        ev["code"] = ev['value'].map(mapping_dict).astype(int)
        ev = ev[ev["code"].isin([0, 1])].reset_index(drop=True)  # using flanker stimulus (both compatible and incompatible) as time-lock

    # from onset second to timestamp under current sampling frequency
    sfreq = raw.info["sfreq"]
    print(f"Current sampling frequency: {sfreq} Hz")
    s1_samples = np.round(ev["onset"].values * sfreq).astype(int)
    events = np.c_[s1_samples, np.zeros_like(s1_samples), np.ones_like(s1_samples).astype(int)]

    # segment epoch
    picks = mne.pick_types(raw.info, eeg=True, eog=False, exclude="bads")
    epochs = mne.Epochs(
        raw, events, event_id=dict(cue=1),
        tmin=tmin, tmax=tmax,
        baseline=baseline, picks=picks,
        proj=False, preload=True, reject=None, verbose=False
    )

    # guarantee int((tmax - tmin) * sfreq) timestamps 
    target_len = int((tmax - tmin) * sfreq)
    data = epochs.get_data()  # (N, C, T)
    if data.shape[-1] > target_len:
        data = data[..., :target_len]
    elif data.shape[-1] < target_len:
        pad = target_len - data.shape[-1]
        data = np.pad(data, ((0, 0), (0, 0), (0, pad)), mode="edge")
    
    # convert to (N, int((tmax - tmin) * sfreq), C)
    X = np.transpose(data, (0, 2, 1))
    
    # generate to Y: [task_id, stimulation_type, subject_id, parent_degree]
    valid_event_idx = epochs.selection   # select the index of event 
    stimulation_type = ev['value'].map(mapping_dict).values[valid_event_idx].astype(int)
    # Create label array y with structure: [disease_id, stimulus_type, subject_id, task_id]
    y = np.column_stack([
        np.full_like(stimulation_type, -1),  # disease_id is not provided, set to -1
        stimulation_type,
        np.full_like(stimulation_type, subject_id),
        np.full_like(stimulation_type, task_id),
    ])
    
    y = y[valid_event_idx]  # y align with X to avoid mismatch
    
    return X, y

In [57]:
import re
import os

def get_subject_id(path: str) -> int:
    # match string like "sub-xxx" 
    match = re.search(r"sub-(\d+)", path)
    if match:
        return int(match.group(1).lstrip("0"))  
    else:
        raise ValueError(f"No subject_id found in path: {path}")

## PASS 1: Find total number of trials across all subjects

In [58]:
N_total_aodd, N_total_vodd, N_total_vs, N_total_flanker = 0, 0, 0, 0
task_id, tmin, tmax, baseline = None, None, None, None
    
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        # print(sub_path)
        subject_id = get_subject_id(sub_path)
        parent_degree_str = participants[participants['participant_id']==sub]['Highest_Adult_Edu_Grouped'].item()
        parent_degree_int = degree_map[parent_degree_str]
        
        for file in os.listdir(sub_path):
            if ('auditoryoddball_eeg.vhdr' in file or 'visualoddball_eeg.vhdr' in file 
                    or "task-visualsearch_eeg.vhdr" in file or "task-flanker_eeg.vhdr" in file): 
                # check task
                # only use visual task here
                if "task-auditoryoddball_eeg.vhdr" in file:
                    task_id = TASK_ID_1
                    tmin = TMIN_1
                    tmax = TMAX_1
                    baseline = BASELINE_1
                if "task-visualoddball_eeg.vhdr" in file: 
                    task_id = TASK_ID_2
                    tmin = TMIN_2
                    tmax = TMAX_2
                    baseline = BASELINE_2
                if "task-visualsearch_eeg.vhdr" in file: 
                    task_id = TASK_ID_3
                    tmin = TMIN_3
                    tmax = TMAX_3
                    baseline = BASELINE_3
                if "task-flanker_eeg.vhdr" in file: 
                    task_id = TASK_ID_4
                    tmin = TMIN_4
                    tmax = TMAX_4
                    baseline = BASELINE_4
                # load eeg data and preprocess
                print(sub_path)
                
                vhdr_path = os.path.join(sub_path, file)
                events_file = file.split(".")[0].split("_")[0] + "_" + file.split(".")[0].split("_")[1] + "_events.tsv"
                events_file_path = os.path.join(sub_path, events_file)
                
                print(vhdr_path)
                raw = mne.io.read_raw_brainvision(vhdr_path, preload=True)                
                raw.pick_types(eeg=True)   # only keep EEG channels
                # raw = data_preprocessing(raw, common_channels, SAMPLE_RATE, notch_freq=60, l_freq=0.5, h_freq=40, verbose=True)
                print()
                
                # segment and make X, y
                
                print(f"Start epoching file {file} making X, y...")
                X, y = epoch_and_make_xy(
                    raw, events_file_path,
                    tmin=tmin, tmax=tmax, baseline=baseline,  #  -200ms - 800ms
                    task_id=task_id, subject_id=subject_id, parent_degree=parent_degree_int, 
                )
                print(f"File {file} epoch into trial X shape: {X.shape}, y shape: {y.shape}")

                if task_id == TASK_ID_1:
                    N_total_aodd += X.shape[0]
                elif task_id == TASK_ID_2:
                    N_total_vodd += X.shape[0]
                elif task_id == TASK_ID_3:
                    N_total_vs += X.shape[0]
                elif task_id == TASK_ID_4:
                    N_total_flanker += X.shape[0]
                
                print("-----------------------\n")
            print("------------------------------------------------\n")

print("Total trials across all subjects in Auditory Oddball:", N_total_aodd)
print("Total trials across all subjects in Visual Oddball:", N_total_vodd)
print("Total trials across all subjects in Visual Search:", N_total_vs)
print("Total trials across all subjects in Flanker Task:", N_total_flanker)

------------------------------------------------

------------------------------------------------

CESCA/sub-001\eeg/
CESCA/sub-001\eeg/sub-001_task-auditoryoddball_eeg.vhdr
Extracting parameters from CESCA/sub-001\eeg/sub-001_task-auditoryoddball_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 124029  =      0.000 ...   248.058 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).

Start epoching file sub-001_task-auditoryoddball_eeg.vhdr making X, y...
Current sampling frequency: 500.0 Hz
File sub-001_task-auditoryoddball_eeg.vhdr epoch into trial X shape: (335, 500, 26), y shape: (335, 4)
-----------------------

------------------------------------------------

------------------------------------------------

------------------------------------------------

------------------------------------------------

------------------------------------------------

------------------------------------------------

CESCA/sub-001\eeg/
CESCA/sub-001

## Create memmap storage for each task

In [59]:
output_root_aodd = os.path.join('Processed', sub_folder_path, 'CESCA-AODD')
os.makedirs(output_root_aodd, exist_ok=True)

X_path_aodd = os.path.join(output_root_aodd, 'X.dat')
y_path_aodd = os.path.join(output_root_aodd, 'y.dat')
meta_path_aodd = os.path.join(output_root_aodd, 'meta.json')

print("X path of aodd task:", X_path_aodd)
print("y path of aodd task:", y_path_aodd)

# create memmap storage
SEG_LEN_AODD = int((TMAX_1 - TMIN_1) * SAMPLE_RATE)  # samples per epoch
n_channels = len(common_channels)
X_mm_aodd = np.memmap(X_path_aodd, dtype='float32', mode='w+', shape=(N_total_aodd, SEG_LEN_AODD, n_channels))
y_mm_aodd = np.memmap(y_path_aodd, dtype='float32', mode='w+', shape=(N_total_aodd, 4))
print("Created memmap for aodd: X shape =", X_mm_aodd.shape, ", y shape =", y_mm_aodd.shape)

X path of aodd task: Processed\200Hz\CESCA-AODD\X.dat
y path of aodd task: Processed\200Hz\CESCA-AODD\y.dat
Created memmap for aodd: X shape = (38151, 200, 26) , y shape = (38151, 4)


In [60]:
output_root_vodd = os.path.join('Processed', sub_folder_path, 'CESCA-VODD')
os.makedirs(output_root_vodd, exist_ok=True)

X_path_vodd = os.path.join(output_root_vodd, 'X.dat')
y_path_vodd = os.path.join(output_root_vodd, 'y.dat')
meta_path_vodd = os.path.join(output_root_vodd, 'meta.json')

print("X path of vodd task:", X_path_vodd)
print("y path of vodd task:", y_path_vodd)

# create memmap storage
SEG_LEN_VODD = int((TMAX_2 - TMIN_2) * SAMPLE_RATE)  # samples per epoch
n_channels = len(common_channels)
X_mm_vodd = np.memmap(X_path_vodd, dtype='float32', mode='w+', shape=(N_total_vodd, SEG_LEN_VODD, n_channels))
y_mm_vodd = np.memmap(y_path_vodd, dtype='float32', mode='w+', shape=(N_total_vodd, 4))
print("Created memmap for vodd: X shape =", X_mm_vodd.shape, ", y shape =", y_mm_vodd.shape)

X path of vodd task: Processed\200Hz\CESCA-VODD\X.dat
y path of vodd task: Processed\200Hz\CESCA-VODD\y.dat
Created memmap for vodd: X shape = (20419, 200, 26) , y shape = (20419, 4)


In [61]:
output_root_vs = os.path.join('Processed', sub_folder_path, 'CESCA-VS')
os.makedirs(output_root_vs, exist_ok=True)

X_path_vs = os.path.join(output_root_vs, 'X.dat')
y_path_vs = os.path.join(output_root_vs, 'y.dat')
meta_path_vs = os.path.join(output_root_vs, 'meta.json')

print("X path of visual search task:", X_path_vs)
print("y path of visual search task:", y_path_vs)

# create memmap storage
SEG_LEN_VS = int((TMAX_3 - TMIN_3) * SAMPLE_RATE)  # samples per epoch
n_channels = len(common_channels)
X_mm_vs = np.memmap(X_path_vs, dtype='float32', mode='w+', shape=(N_total_vs, SEG_LEN_VS, n_channels))
y_mm_vs = np.memmap(y_path_vs, dtype='float32', mode='w+', shape=(N_total_vs, 4))
print("Created memmap for visual search task: X shape =", X_mm_vs.shape, ", y shape =", y_mm_vs.shape)

X path of visual search task: Processed\200Hz\CESCA-VS\X.dat
y path of visual search task: Processed\200Hz\CESCA-VS\y.dat
Created memmap for visual search task: X shape = (23706, 200, 26) , y shape = (23706, 4)


In [62]:
output_root_flanker = os.path.join('Processed', sub_folder_path, 'CESCA-FLANKER')
os.makedirs(output_root_flanker, exist_ok=True)

X_path_flanker = os.path.join(output_root_flanker, 'X.dat')
y_path_flanker = os.path.join(output_root_flanker, 'y.dat')
meta_path_flanker = os.path.join(output_root_flanker, 'meta.json')

print("X path of flanker task:", X_path_flanker)
print("y path of flanker task:", y_path_flanker)

# create memmap storage
SEG_LEN_FLANKER = int((TMAX_4 - TMIN_4) * SAMPLE_RATE)  # samples per epoch
n_channels = len(common_channels)
X_mm_flanker = np.memmap(X_path_flanker, dtype='float32', mode='w+', shape=(N_total_flanker, SEG_LEN_FLANKER, n_channels))
y_mm_flanker = np.memmap(y_path_flanker, dtype='float32', mode='w+', shape=(N_total_flanker, 4))
print("Created memmap for flanker task: X shape =", X_mm_flanker.shape, ", y shape =", y_mm_flanker.shape)

X path of flanker task: Processed\200Hz\CESCA-FLANKER\X.dat
y path of flanker task: Processed\200Hz\CESCA-FLANKER\y.dat
Created memmap for flanker task: X shape = (29774, 200, 26) , y shape = (29774, 4)


## PASS 2: Process and store trials into memmap

In [63]:
cur_aodd_idx, cur_vodd_idx, cur_vs_idx, cur_flanker_idx = 0, 0, 0, 0
    
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        # print(sub_path)
        subject_id = get_subject_id(sub_path)
        parent_degree_str = participants[participants['participant_id']==sub]['Highest_Adult_Edu_Grouped'].item()
        parent_degree_int = degree_map[parent_degree_str]
        
        for file in os.listdir(sub_path):
            if ('auditoryoddball_eeg.vhdr' in file or 'visualoddball_eeg.vhdr' in file 
                    or "task-visualsearch_eeg.vhdr" in file or "task-flanker_eeg.vhdr" in file): 
                # check task
                # only use visual task here
                if "task-auditoryoddball_eeg.vhdr" in file:
                    task_id = TASK_ID_1
                    tmin = TMIN_1
                    tmax = TMAX_1
                    baseline = BASELINE_1
                if "task-visualoddball_eeg.vhdr" in file: 
                    task_id = TASK_ID_2
                    tmin = TMIN_2
                    tmax = TMAX_2
                    baseline = BASELINE_2
                if "task-visualsearch_eeg.vhdr" in file: 
                    task_id = TASK_ID_3
                    tmin = TMIN_3
                    tmax = TMAX_3
                    baseline = BASELINE_3
                if "task-flanker_eeg.vhdr" in file: 
                    task_id = TASK_ID_4
                    tmin = TMIN_4
                    tmax = TMAX_4
                    baseline = BASELINE_4
                # load eeg data and preprocess
                print(sub_path)
                
                vhdr_path = os.path.join(sub_path, file)
                events_file = file.split(".")[0].split("_")[0] + "_" + file.split(".")[0].split("_")[1] + "_events.tsv"
                events_file_path = os.path.join(sub_path, events_file)
                
                print(vhdr_path)
                raw = mne.io.read_raw_brainvision(vhdr_path, preload=True)                
                raw.pick_types(eeg=True)   # only keep EEG channels
                raw = data_preprocessing(raw, common_channels, SAMPLE_RATE, notch_freq=60, l_freq=0.5, h_freq=40, verbose=True)
                print()
                
                # segment and make X, y
                
                print(f"Start epoching file {file} making X, y...")
                X, y = epoch_and_make_xy(
                    raw, events_file_path,
                    tmin=tmin, tmax=tmax, baseline=baseline,  #  -200ms - 800ms
                    task_id=task_id, subject_id=subject_id, parent_degree=parent_degree_int, 
                )
                print(f"File {file} epoch into trial X shape: {X.shape}, y shape: {y.shape}")

                if task_id == TASK_ID_1:
                    X_mm_aodd[cur_aodd_idx:cur_aodd_idx+X.shape[0]] = X
                    y_mm_aodd[cur_aodd_idx:cur_aodd_idx+y.shape[0]] = y
                    cur_aodd_idx += X.shape[0]
                elif task_id == TASK_ID_2:
                    X_mm_vodd[cur_vodd_idx:cur_vodd_idx+X.shape[0]] = X
                    y_mm_vodd[cur_vodd_idx:cur_vodd_idx+y.shape[0]] = y
                    cur_vodd_idx += X.shape[0]
                elif task_id == TASK_ID_3:
                    X_mm_vs[cur_vs_idx:cur_vs_idx+X.shape[0]] = X
                    y_mm_vs[cur_vs_idx:cur_vs_idx+y.shape[0]] = y
                    cur_vs_idx += X.shape[0]
                elif task_id == TASK_ID_4:
                    X_mm_flanker[cur_flanker_idx:cur_flanker_idx+X.shape[0]] = X
                    y_mm_flanker[cur_flanker_idx:cur_flanker_idx+y.shape[0]] = y
                    cur_flanker_idx += X.shape[0]
                
                print("-----------------------\n")
            print("------------------------------------------------\n")

print("DONE: cur auditory oddball =", cur_aodd_idx, " expected N_total_oddball =", N_total_aodd)
print("DONE: cur visual oddball =", cur_vodd_idx, " expected N_total_oddball =", N_total_vodd)
print("DONE: cur visual search =", cur_vs_idx, " expected N_total_visual search =", N_total_vs)
print("DONE: cur flanker =", cur_flanker_idx, " expected N_total_flanker =", N_total_flanker)
      
del X_mm_aodd, y_mm_aodd, X_mm_vodd, y_mm_vodd, X_mm_vs, y_mm_vs, X_mm_flanker, y_mm_flanker

------------------------------------------------

------------------------------------------------

CESCA/sub-001\eeg/
CESCA/sub-001\eeg/sub-001_task-auditoryoddball_eeg.vhdr
Extracting parameters from CESCA/sub-001\eeg/sub-001_task-auditoryoddball_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 124029  =      0.000 ...   248.058 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
✔ Step 1: Picked common channels (26): ['F7', 'FC1', 'P8', 'FC5', 'P4', 'CP1', 'CP6', 'C4', 'P7', 'Oz', 'F3', 'FC2', 'CP5', 'Fp2', 'O2', 'Pz', 'T7', 'P3', 'FC6', 'F4', 'Fp1', 'O1', 'C3', 'CP2', 'F8', 'Fz']
✔ Step 2, Montage set: 'standard_1020'.
✔ Step 3: Notch @ 60 Hz
✔ Step 4: Band-pass 0.5–40 Hz
ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)
✔ Step 6: Average reference

Fitting ICA to data using 26 channels (please be patient, this may take a while)
Selecting

## Create JSON file

In [64]:
meta_aodd = {
    "N": int(N_total_aodd),
    "T": SEG_LEN_AODD,
    "C": int(n_channels),
    "SAMPLE_RATE": SAMPLE_RATE,
    "TMIN": TMIN_1,
    "TMAX": TMAX_1,
    "BASELINE": BASELINE_1,
    "MONTAGE": "standard_1020",
    "CHANNELS": common_channels,
    "LABELS": {
        "disease_id": {-1: "unknown"},
        "stimulus_type": {0: "Standard stimulus", 1: "Target stimulus"},
        "subject_id": "integer from 1 to number of subjects",
        "task_id": {0: "auditory oddball"}
    }
}
with open(meta_path_aodd, "w") as f:
    json.dump(meta_aodd, f, indent=2)

print("Saved meta:", meta_path_aodd)

Saved meta: Processed\200Hz\CESCA-AODD\meta.json


In [65]:
meta_vodd = {
    "N": int(N_total_vodd),
    "T": SEG_LEN_VODD,
    "C": int(n_channels),
    "SAMPLE_RATE": SAMPLE_RATE,
    "TMIN": TMIN_2,
    "TMAX": TMAX_2,
    "BASELINE": BASELINE_2,
    "MONTAGE": "standard_1020",
    "CHANNELS": common_channels,
    "LABELS": {
        "disease_id": {-1: "unknown"},
        "stimulus_type": {0: "Standard stimulus", 1: "Target stimulus"},
        "subject_id": "integer from 1 to number of subjects",
        "task_id": {1: "visual oddball"}
    }
}
with open(meta_path_vodd, "w") as f:
    json.dump(meta_vodd, f, indent=2)
    
print("Saved meta:", meta_path_vodd)

Saved meta: Processed\200Hz\CESCA-VODD\meta.json


In [66]:
meta_vs = {
    "N": int(N_total_vs),
    "T": SEG_LEN_VS,
    "C": int(n_channels),
    "SAMPLE_RATE": SAMPLE_RATE,
    "TMIN": TMIN_3,
    "TMAX": TMAX_3,
    "BASELINE": BASELINE_3,
    "MONTAGE": "standard_1020",
    "CHANNELS": common_channels,
    "LABELS": {
        "disease_id": {-1: "unknown"},
        "stimulus_type": {0: "Target left", 1: "Target right"},
        "subject_id": "integer from 1 to number of subjects",
        "task_id": {10: "visual search"}
    }
}

with open(meta_path_vs, "w") as f:
    json.dump(meta_vs, f, indent=2)
    
print("Saved meta:", meta_path_vs)

Saved meta: Processed\200Hz\CESCA-VS\meta.json


In [67]:
meta_flanker = {
    "N": int(N_total_flanker),
    "T": SEG_LEN_FLANKER,
    "C": int(n_channels),
    "SAMPLE_RATE": SAMPLE_RATE,
    "TMIN": TMIN_4,
    "TMAX": TMAX_4,
    "BASELINE": BASELINE_4,
    "MONTAGE": "standard_1020",
    "CHANNELS": common_channels,
    "LABELS": {
        "disease_id": {-1: "unknown"},
        "stimulus_type": {0: "Compatible", 1: "Incompatible"},
        "subject_id": "integer from 1 to number of subjects",
        "task_id": {5: "flanker task"}
    }
}

with open(meta_path_flanker, "w") as f:
    json.dump(meta_flanker, f, indent=2)

print("Saved meta:", meta_path_flanker)

Saved meta: Processed\200Hz\CESCA-FLANKER\meta.json


## Load and check the processed data

In [68]:
# load meta information
meta_path = meta_path_aodd  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path_aodd, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path_aodd, dtype='float32', mode='r', shape=(N, 4))

# subject_id is stored in y[:, 2]
subject_ids = np.unique(y_mm[:, 2]).astype(int)

total_trials = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 2] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 4)       # logical y shape for this subject
    total_trials += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total trials:", total_trials)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 38151
T: 200
C: 26
SAMPLE_RATE: 200
TMIN: -0.2
TMAX: 0.8
BASELINE: [-0.2, 0]
MONTAGE: standard_1020
CHANNELS: ['F7', 'FC1', 'P8', 'FC5', 'P4', 'CP1', 'CP6', 'C4', 'P7', 'Oz', 'F3', 'FC2', 'CP5', 'Fp2', 'O2', 'Pz', 'T7', 'P3', 'FC6', 'F4', 'Fp1', 'O1', 'C3', 'CP2', 'F8', 'Fz']
LABELS: {'disease_id': {'-1': 'unknown'}, 'stimulus_type': {'0': 'Standard stimulus', '1': 'Target stimulus'}, 'subject_id': 'integer from 1 to number of subjects', 'task_id': {'0': 'auditory oddball'}}
-------------------------------------
Subject ID 001: X shape=(335, 200, 26), y shape=(335, 4)
Subject ID 013: X shape=(335, 200, 26), y shape=(335, 4)
Subject ID 014: X shape=(335, 200, 26), y shape=(335, 4)
Subject ID 015: X shape=(335, 200, 26), y shape=(335, 4)
Subject ID 016: X shape=(335, 200, 26), y shape=(335, 4)
Subject ID 017: X shape=(333, 200, 26), y shape=(333, 4)
Subject ID 018: X shape=(335, 200, 26), y shape=(335, 4)
Subject ID 020: X shape=(335, 200, 26), y shape=(335, 4)
Subject ID 021: X

In [69]:
# load meta information
meta_path = meta_path_vodd  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path_vodd, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path_vodd, dtype='float32', mode='r', shape=(N, 4))

# subject_id is stored in y[:, 2]
subject_ids = np.unique(y_mm[:, 2]).astype(int)

total_trials = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 2] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 4)       # logical y shape for this subject
    total_trials += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total trials:", total_trials)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 20419
T: 200
C: 26
SAMPLE_RATE: 200
TMIN: -0.2
TMAX: 0.8
BASELINE: [-0.2, 0]
MONTAGE: standard_1020
CHANNELS: ['F7', 'FC1', 'P8', 'FC5', 'P4', 'CP1', 'CP6', 'C4', 'P7', 'Oz', 'F3', 'FC2', 'CP5', 'Fp2', 'O2', 'Pz', 'T7', 'P3', 'FC6', 'F4', 'Fp1', 'O1', 'C3', 'CP2', 'F8', 'Fz']
LABELS: {'disease_id': {'-1': 'unknown'}, 'stimulus_type': {'0': 'Standard stimulus', '1': 'Target stimulus'}, 'subject_id': 'integer from 1 to number of subjects', 'task_id': {'1': 'visual oddball'}}
-------------------------------------
Subject ID 001: X shape=(210, 200, 26), y shape=(210, 4)
Subject ID 002: X shape=(210, 200, 26), y shape=(210, 4)
Subject ID 003: X shape=(210, 200, 26), y shape=(210, 4)
Subject ID 004: X shape=(210, 200, 26), y shape=(210, 4)
Subject ID 005: X shape=(210, 200, 26), y shape=(210, 4)
Subject ID 006: X shape=(210, 200, 26), y shape=(210, 4)
Subject ID 007: X shape=(210, 200, 26), y shape=(210, 4)
Subject ID 008: X shape=(210, 200, 26), y shape=(210, 4)
Subject ID 009: X s

In [70]:
# load meta information
meta_path = meta_path_vs  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path_vs, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path_vs, dtype='float32', mode='r', shape=(N, 4))

# subject_id is stored in y[:, 2]
subject_ids = np.unique(y_mm[:, 2]).astype(int)

total_trials = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 2] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 4)       # logical y shape for this subject
    total_trials += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total trials:", total_trials)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 23706
T: 200
C: 26
SAMPLE_RATE: 200
TMIN: -0.2
TMAX: 0.8
BASELINE: [-0.2, 0]
MONTAGE: standard_1020
CHANNELS: ['F7', 'FC1', 'P8', 'FC5', 'P4', 'CP1', 'CP6', 'C4', 'P7', 'Oz', 'F3', 'FC2', 'CP5', 'Fp2', 'O2', 'Pz', 'T7', 'P3', 'FC6', 'F4', 'Fp1', 'O1', 'C3', 'CP2', 'F8', 'Fz']
LABELS: {'disease_id': {'-1': 'unknown'}, 'stimulus_type': {'0': 'Target left', '1': 'Target right'}, 'subject_id': 'integer from 1 to number of subjects', 'task_id': {'10': 'visual search'}}
-------------------------------------
Subject ID 001: X shape=(330, 200, 26), y shape=(330, 4)
Subject ID 002: X shape=(330, 200, 26), y shape=(330, 4)
Subject ID 003: X shape=(330, 200, 26), y shape=(330, 4)
Subject ID 004: X shape=(330, 200, 26), y shape=(330, 4)
Subject ID 005: X shape=(330, 200, 26), y shape=(330, 4)
Subject ID 006: X shape=(305, 200, 26), y shape=(305, 4)
Subject ID 007: X shape=(330, 200, 26), y shape=(330, 4)
Subject ID 008: X shape=(330, 200, 26), y shape=(330, 4)
Subject ID 009: X shape=(330

In [71]:
# load meta information
meta_path = meta_path_flanker  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path_flanker, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path_flanker, dtype='float32', mode='r', shape=(N, 4))

# subject_id is stored in y[:, 2]
subject_ids = np.unique(y_mm[:, 2]).astype(int)

total_trials = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 2] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 4)       # logical y shape for this subject
    total_trials += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total trials:", total_trials)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 29774
T: 200
C: 26
SAMPLE_RATE: 200
TMIN: -0.2
TMAX: 0.8
BASELINE: [-0.2, 0]
MONTAGE: standard_1020
CHANNELS: ['F7', 'FC1', 'P8', 'FC5', 'P4', 'CP1', 'CP6', 'C4', 'P7', 'Oz', 'F3', 'FC2', 'CP5', 'Fp2', 'O2', 'Pz', 'T7', 'P3', 'FC6', 'F4', 'Fp1', 'O1', 'C3', 'CP2', 'F8', 'Fz']
LABELS: {'disease_id': {'-1': 'unknown'}, 'stimulus_type': {'0': 'Compatible', '1': 'Incompatible'}, 'subject_id': 'integer from 1 to number of subjects', 'task_id': {'5': 'flanker task'}}
-------------------------------------
Subject ID 001: X shape=(410, 200, 26), y shape=(410, 4)
Subject ID 002: X shape=(410, 200, 26), y shape=(410, 4)
Subject ID 003: X shape=(410, 200, 26), y shape=(410, 4)
Subject ID 004: X shape=(410, 200, 26), y shape=(410, 4)
Subject ID 005: X shape=(410, 200, 26), y shape=(410, 4)
Subject ID 006: X shape=(410, 200, 26), y shape=(410, 4)
Subject ID 007: X shape=(410, 200, 26), y shape=(410, 4)
Subject ID 008: X shape=(410, 200, 26), y shape=(410, 4)
Subject ID 009: X shape=(410, 2